In [2]:
draft_pick_value = pd.read_csv('/Users/austinriley/Downloads/Warriors Models:Analytics/draft_pick_no_value.csv')
team_rating_data_new = pd.read_csv('NBA_team_rating_data_2000-2026.csv')
nba_drafts = pd.read_csv('/Users/austinriley/Downloads/Warriors Models:Analytics/NBA_draft_classes_2016-2026.csv')
nba_drafts = nba_drafts[nba_drafts['draft_year'] >= 2025]
nba_draft_history = nba_drafts[['team_id','pick_overall','draft_year']].drop_duplicates()
nba_draft_history.loc[nba_draft_history['team_id'] == 'BRK', 'team_id'] = 'BKN'
nba_draft_history.loc[nba_draft_history['team_id'] == 'CHO', 'team_id'] = 'CHA'
nba_draft_history.loc[nba_draft_history['team_id'] == 'PHO', 'team_id'] = 'PHX'
first_pick_26 = nba_draft_history[(nba_draft_history['draft_year'] == 2026) & (nba_draft_history['pick_overall'] == 1)]['team_id']
top_five_25_26 = pd.merge(nba_draft_history[(nba_draft_history['draft_year'] == 2026) & (nba_draft_history['pick_overall'] <= 5)],nba_draft_history[(nba_draft_history['draft_year'] == 2025) & (nba_draft_history['pick_overall'] <= 5)], how='inner',on='team_id')['team_id']
draft_pick_value

,Unnamed: 0,pick_overall,value
0,0,1,4.050000
1,1,2,3.883645
2,2,3,3.786333
3,3,4,3.717289
4,4,5,3.663735
5,5,6,3.619978
6,6,7,3.582982
7,7,8,3.550934
8,8,9,3.522666
9,9,10,3.497380


In [1]:
import pandas as pd
import numpy as np
import warnings
import unicodedata
import statsmodels.api as sm
warnings.filterwarnings('ignore')
import plotly.io as pio
import plotly.express as px
def clean_name(s):
    if not isinstance(s, str):
        return s
    try:
        s = s.encode('latin1').decode('utf-8')  # fix mojibake, if present
    except (UnicodeDecodeError, UnicodeEncodeError):
        pass
    normalized = unicodedata.normalize('NFKD', s)
    return ''.join(c for c in normalized if not unicodedata.combining(c))

In [3]:
lottery_balls = pd.read_csv('NBA Draft Lottery Pick Value - New Lottery Balls.csv')
lottery_balls.head()

,Worst Record,Lottery Balls
0,1,1
1,1,1
2,2,1
3,2,1
4,3,1


In [4]:
team_draft_picks = pd.read_csv('NBA Draft Lottery Pick Value - NBA_team_draft_picks.csv')
team_draft_picks

,originating_team,year,round,rights_holder,notes
0,ATL,2027,1,SAS,No protections
1,ATL,2027,2,ATL,NaN
2,ATL,2028,1,ATL,NaN
3,ATL,2028,2,BKN,No protections
4,ATL,2029,1,ATL,NaN
...,...,...,...,...,...
419,WAS,2031,2,LAL,No protections
420,WAS,2032,1,WAS,NaN
421,WAS,2032,2,LAL,No protections
422,WAS,2033,1,WAS,NaN


In [5]:
team_rating_data_new = team_rating_data_new.sort_values(by=['team_name','season'])
team_rating_data_new['next_yr_win_loss'] = team_rating_data_new.groupby('team_name')['win_loss_pct'].shift(-1)
team_rating_data_new['toward_500'] = (team_rating_data_new['next_yr_win_loss'] - .5000) - (team_rating_data_new['win_loss_pct']  - .5000)
team_rating_data_new = team_rating_data_new[['team_name', 'wins',
       'losses', 'win_loss_pct', 'mov', 'season',
       'next_yr_win_loss', 'toward_500']].dropna()
team_rating_data_new.head()

,team_name,wins,losses,win_loss_pct,mov,season,next_yr_win_loss,toward_500
25,Atlanta Hawks,28,54,0.341,-5.38,2000,0.305,-0.036
54,Atlanta Hawks,25,57,0.305,-5.21,2001,0.402,0.097
81,Atlanta Hawks,33,49,0.402,-4.23,2002,0.427,0.025
110,Atlanta Hawks,35,47,0.427,-3.56,2003,0.341,-0.086
141,Atlanta Hawks,28,54,0.341,-4.65,2004,0.159,-0.182


In [6]:
%%sql
SELECT * FROM public.player_season_epm

,id,season,age,seasontype,team_id,team_alias,team_alias_all,player_id,player_name,pos_text,...,drbpct,astpct,topct,stlpct,blkpct,fga_rim_75,fga_mid_75,fg3a_75,fta_75,fga_75
0,1,2009-10,25,2,1610612739,CLE,CLE,2544,LeBron James,SF,...,0.184140,0.398529,0.123346,0.022222,0.020047,7.296080,7.81250,5.124470,10.209200,20.23310
1,2,2009-10,21,2,1610612760,OKC,OKC,201142,Kevin Durant,PF,...,0.178313,0.129432,0.118083,0.017680,0.018987,5.081140,10.45770,4.147630,9.878680,19.68650
2,3,2009-10,27,2,1610612748,MIA,MIA,2548,Dwyane Wade,SG,...,0.110373,0.333777,0.122330,0.026662,0.023617,8.217640,9.62477,3.419320,9.765480,21.26170
3,4,2009-10,24,2,1610612753,ORL,ORL,2730,Dwight Howard,C,...,0.315644,0.086852,0.184885,0.013948,0.060622,7.494460,3.96195,0.096971,11.193200,11.55340
4,5,2009-10,24,2,1610612740,NOH,NOH,101108,Chris Paul,PG,...,0.118421,0.456274,0.135103,0.029575,0.003493,3.182660,8.57934,2.928970,4.335790,14.69100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8109,8110,2025-26,26,2,1610612757,POR,POR,1631451,Javonte Cooke,SG,...,0.185185,0.127273,0.176471,0.029126,0.000000,3.605770,2.88462,8.293270,0.721154,14.78370
8110,8111,2025-26,31,2,1610612756,PHX,PHX,1628502,Nigel Hayes-Davis,SF,...,0.125000,0.054688,0.142857,0.020050,0.009662,0.755668,4.91184,3.022670,0.755668,8.69018
8111,8112,2025-26,28,2,1610612741,CHI,"CHI,PHX",1630208,Nick Richards,C,...,0.251135,0.032520,0.193443,0.006775,0.040842,6.766920,2.61449,0.922761,5.177720,10.30420
8112,8113,2025-26,21,2,1610612742,DAL,"DAL,WAS",1642358,AJ Johnson,SG,...,0.101737,0.154341,0.133929,0.011628,0.006048,7.065220,2.17391,4.425470,3.105590,13.66460


In [7]:
# ============================================================
# EPM (external source — SQL query result, must be loaded before this runs)
# SELECT * FROM public.player_season_epm
# ============================================================
# epm = <load from your SQL source here>

epm['season'] = ('20' + epm['season'].astype(str).str[-2:]).astype(int)
epm['exp'] = epm['season'].astype(int) - epm['rookie_year'].astype(int)
epm = epm.sort_values(by=['player_name', 'season'])
epm['prev_year_mpg'] = epm['mpg'].shift(1, fill_value=0)
epm['player_name'] = epm['player_name'].str.lower()
epm['player_name'] = epm['player_name'].str.replace('.', '')


In [8]:

# ============================================================
# Advanced stats (2017-2026)
# ============================================================
advanced_stats_df = pd.read_csv('NBA_advanced_stats_2000-2026')
advanced_stats_df['name_display'] = advanced_stats_df['name_display'].apply(clean_name)
multi_team_labels = {'2TM', '3TM', '4TM', '5TM'}

# For each (player, season) group, check if an aggregate row exists
has_agg = advanced_stats_df.groupby(['name_display', 'season'])['team_name_abbr'].transform(
    lambda s: s.isin(multi_team_labels).any()
)

# Rows to drop: belongs to a group with an aggregate row, AND is itself a single-team row
is_individual_team_row = ~advanced_stats_df['team_name_abbr'].isin(multi_team_labels)
drop_mask = has_agg & is_individual_team_row

advanced_stats_df = advanced_stats_df[~drop_mask].copy()
advanced_stats_df = advanced_stats_df[['name_display', 'team_name_abbr', 'season', 'mp', 'per', 'ws', 'bpm']]
advanced_stats_df = advanced_stats_df[advanced_stats_df['mp'] >= 200]
advanced_stats_df['zs_bpm'] = advanced_stats_df.groupby('season')['bpm'].transform(lambda x: (x - x.min()) * 10 /(x.max() - x.min()))
advanced_stats_df['zs_per'] = advanced_stats_df.groupby('season')['per'].transform(lambda x: (x - x.min()) * 10 /(x.max() - x.min()))
advanced_stats_df['zs_ws'] = advanced_stats_df.groupby('season')['ws'].transform(lambda x: (x - x.min()) * 10 /(x.max() - x.min()))
advanced_stats_df = advanced_stats_df[~advanced_stats_df['name_display'].isna()]
advanced_stats_df = advanced_stats_df.sort_values(by=['name_display', 'season'])
advanced_stats_df = advanced_stats_df[advanced_stats_df['name_display'] != 'Justin Jackson']
advanced_stats_df['name_display'] = advanced_stats_df['name_display'].str.lower()
advanced_stats_df['name_display'] = advanced_stats_df['name_display'].str.replace('.', '')


In [9]:

# ============================================================
# Draft classes (2016-2026)
# ============================================================
draft = pd.read_csv('/Users/austinriley/Downloads/Warriors Models:Analytics/NBA_draft_classes_2000-2026.csv')
draft['first_season'] = draft['draft_year'] + 1
draft['player'] = draft['player'].str.lower()
draft['player'] = draft['player'].str.replace('.', '')


In [10]:

# ============================================================
# Rookie-year EPM by pick (used to impute rookie/unproven-player value)
# ============================================================
rookie_epm = pd.merge(draft, epm[['player_name', 'season', 'tot']], how='left', left_on=['player', 'first_season'], right_on=['player_name', 'season'])
rookie_epm = pd.merge(rookie_epm, advanced_stats_df, how='left', left_on=['player', 'first_season'], right_on=['name_display', 'season'])
rookie_epm['zs_epm'] = rookie_epm.groupby('season_x')['tot'].transform(lambda x: (x - x.min()) * 10 /(x.max() - x.min()))
rookie_epm['advanced_stat_score'] = (0.25 * rookie_epm['zs_epm']) + (0.15 * rookie_epm['zs_bpm']) + (0.30 * rookie_epm['zs_per']) + (0.30 * rookie_epm['zs_ws'])
first_year_epm_by_pick = rookie_epm.groupby('pick_overall').agg({'tot': 'median', 'advanced_stat_score': 'median'}).reset_index()
first_year_epm_by_pick

,pick_overall,tot,advanced_stat_score
0,1.0,-0.932834,4.685464
1,2.0,-1.722580,3.564291
2,3.0,-1.853020,4.058256
3,4.0,-1.442830,3.729197
4,5.0,-2.609020,2.355131
5,6.0,-2.333800,3.390104
6,7.0,-2.298455,3.074746
7,8.0,-2.168950,3.176544
8,9.0,-2.277340,3.102877
9,10.0,-2.112505,3.138760


In [11]:

# ============================================================
# Full player box scores (2023-24, 2024-25, 2025-26)
# ============================================================
player_box_score_full_df_2324 = pd.read_csv('/Users/austinriley/Downloads/Warriors Models:Analytics/NBA_player_box_scores_2023-24.csv')
player_box_score_full_df_2324['season'] = '2024'
player_box_score_full_df_2425 = pd.read_csv('/Users/austinriley/Downloads/Warriors Models:Analytics/NBA_player_box_scores_2024-25.csv')
player_box_score_full_df_2425['season'] = '2025'
player_box_score_full_df_2526 = pd.read_csv('/Users/austinriley/Downloads/Warriors Models:Analytics/NBA_player_box_scores_2025-26.csv')
player_box_score_full_df_2526['season'] = '2026'
player_box_score_full_df_2526['name_display'] = (player_box_score_full_df_2526['firstName'] + ' ' + player_box_score_full_df_2526['familyName']).apply(clean_name)

full_player_box_score_df = pd.concat([player_box_score_full_df_2324, player_box_score_full_df_2425, player_box_score_full_df_2526])
full_player_box_score_df['name_display'] = (full_player_box_score_df['firstName'] + ' ' + full_player_box_score_df['familyName']).apply(clean_name)
full_player_box_score_df['season'] = full_player_box_score_df['season'].astype(int)
full_player_box_score_df['name_display'] = full_player_box_score_df['name_display'].str.lower()
full_player_box_score_df['name_display'] = full_player_box_score_df['name_display'].str.replace('.', '')


In [12]:

# ============================================================
# Full TEAM box scores (2023-24, 2024-25, 2025-26)
# ============================================================
box_score_full_df_2324 = pd.read_csv('team_box_score_full_df_2023-24.csv')
box_score_full_df_2324['season'] = '2024'
box_score_full_df_2425 = pd.read_csv('team_box_score_full_df_2024-25.csv')
box_score_full_df_2425['season'] = '2025'
box_score_full_df_2526 = pd.read_csv('team_box_score_full_df_2025-26.csv')
box_score_full_df_2526['season'] = '2026'
full_box_score_df = pd.concat([box_score_full_df_2324, box_score_full_df_2425, box_score_full_df_2526])


In [13]:

# ============================================================
# 2026-27 schedule -> long format (one row per team per game) + is_home flag
# ============================================================
nba_schedule_27 = pd.read_csv('/Users/austinriley/Downloads/NBA Player_Team Info 2025-26 - 2026-27 Schedule.csv')
nba_schedule_27.columns = nba_schedule_27.columns.str.lower()

all_team_games = pd.concat([
    nba_schedule_27[['date', 'home tricode']].rename(columns={'home tricode': 'team tricode'}),
    nba_schedule_27[['date', 'visitor tricode']].rename(columns={'visitor tricode': 'team tricode'})
])
all_team_games = pd.merge(all_team_games, nba_schedule_27[['date', 'home tricode', 'visitor tricode']], how='left', left_on=['date', 'team tricode'], right_on=['date', 'home tricode'])
all_team_games = pd.merge(all_team_games, nba_schedule_27[['date', 'home tricode', 'visitor tricode']], how='left', left_on=['date', 'team tricode'], right_on=['date', 'visitor tricode'])
all_team_games['is_home'] = np.where((all_team_games['team tricode'] == all_team_games['home tricode_x']) | (all_team_games['team tricode'] == all_team_games['home tricode_y']), 1, 0)
all_team_games['home tricode'] = all_team_games[['home tricode_x', 'home tricode_y']].bfill(axis=1).iloc[:, 0]
all_team_games['visitor tricode'] = all_team_games[['visitor tricode_x', 'visitor tricode_y']].bfill(axis=1).iloc[:, 0]
all_team_games['opponent tricode'] = np.where(all_team_games['team tricode'] == all_team_games['home tricode'], all_team_games['visitor tricode'], all_team_games['home tricode'])
nba_schedule_27 = all_team_games[['date', 'team tricode', 'opponent tricode', 'is_home']]


In [14]:

# ============================================================
# Schedule enrichment: is_b2b, rows_ago, gameid, season_game_no
# ============================================================
nba_schedule_27['date'] = pd.to_datetime(nba_schedule_27['date'])
nba_schedule_27['date'] = nba_schedule_27['date'].dt.date
nba_schedule_27 = nba_schedule_27.sort_values(by=['team tricode', 'date'], ascending=True)

team = nba_schedule_27['team tricode'].drop_duplicates()

schedule_27_box = []
for team in team:
    temp_df = nba_schedule_27[(nba_schedule_27['team tricode'] == team)]
    temp_df['days_between'] = temp_df.groupby(['team tricode'])['date'].diff()
    temp_df['two_games_ago'] = temp_df['date'] - temp_df['date'].shift(2)
    temp_df['three_games_ago'] = temp_df['date'] - temp_df['date'].shift(3)
    is_target = (temp_df["is_home"] == True).astype(int)
    temp_df["rows_ago"] = (
        is_target.groupby((is_target == 1).cumsum()).cumcount()
    )
    schedule_27_box.append(temp_df)

schedule_27_box_df = pd.concat(schedule_27_box)
schedule_27_box_df['is_b2b'] = np.where(schedule_27_box_df['days_between'] <= pd.to_timedelta(1, unit='D'), 1, 0)
schedule_27_box_df['is_home'] = schedule_27_box_df['is_home'].astype(int)
schedule_27_box_df['gameid'] = schedule_27_box_df['date'].astype(str) + schedule_27_box_df['team tricode'] + schedule_27_box_df['opponent tricode']
schedule_27_box_df['season_game_no'] = schedule_27_box_df.groupby(['team tricode'])['date'].rank(ascending=True)
schedule_27_box_df.head()


,date,team tricode,opponent tricode,is_home,days_between,two_games_ago,three_games_ago,rows_ago,is_b2b,gameid,season_game_no
1204,2026-10-21,ATL,ORL,0,NaN,NaN,NaN,0,0,2026-10-21ATLORL,1.0
1219,2026-10-23,ATL,CHA,0,"2 days, 0:00:00",NaN,NaN,1,0,2026-10-23ATLCHA,2.0
33,2026-10-24,ATL,HOU,1,"1 day, 0:00:00","3 days, 0:00:00",NaN,0,1,2026-10-24ATLHOU,3.0
1249,2026-10-26,ATL,HOU,0,"2 days, 0:00:00","3 days, 0:00:00","5 days, 0:00:00",1,0,2026-10-26ATLHOU,4.0
56,2026-10-28,ATL,MIA,1,"2 days, 0:00:00","4 days, 0:00:00","5 days, 0:00:00",0,0,2026-10-28ATLMIA,5.0


In [15]:

# ============================================================
# 2026-27 roster + games-missed / durability estimate
# ============================================================
nba_roster_27 = pd.read_csv('/Users/austinriley/Downloads/Warriors Models:Analytics/NBA Player_Team Info 2025-26 - 2026-27_NBA_rosters.csv')

player_totals = pd.read_csv('/Users/austinriley/Downloads/Warriors Models:Analytics/NBA_player_totals_2000-2026.csv')
player_totals['name_display'] = player_totals['name_display'].apply(clean_name)
multi_team_labels = {'2TM', '3TM', '4TM', '5TM'}

has_agg = player_totals.groupby(['name_display', 'season'])['team_name_abbr'].transform(
    lambda s: s.isin(multi_team_labels).any()
)
is_individual_team_row = ~player_totals['team_name_abbr'].isin(multi_team_labels)
drop_mask = has_agg & is_individual_team_row
player_totals = player_totals[~drop_mask].copy()
player_totals['szn_ago'] = player_totals.groupby('name_display')['season'].rank(ascending=False, method='dense')

player_missed_game_pct = player_totals[player_totals['szn_ago'] <= 3]
player_missed_game_pct['gp_pct'] = player_missed_game_pct['games'] / 82
player_missed_game_pct = player_missed_game_pct.groupby('name_display').agg({'gp_pct': 'median'}).reset_index()

nba_roster_27_missed_game = pd.merge(nba_roster_27, player_missed_game_pct[['name_display', 'gp_pct']], how='left', left_on='player', right_on='name_display')
nba_roster_27_missed_game.fillna(0.75, inplace=True)
nba_roster_27_missed_game = nba_roster_27_missed_game[['player', 'pos', 'team', 'pre_season_injury_flag', 'est_pct_season_missed', 'gp_pct', 'Team MPG Rank']]
nba_roster_27_missed_game.loc[nba_roster_27_missed_game['gp_pct'] == 1.0000, 'gp_pct'] = .9900
nba_roster_27_missed_game['player'] = nba_roster_27_missed_game['player'].replace('.', '')
nba_roster_27_missed_game['player'] = nba_roster_27_missed_game['player'].str.lower()


In [16]:

# ============================================================
# Most recent EPM per player (for established players' value)
# ============================================================
epm_recent = epm[epm['season'] >= 2020][['season', 'player_name', 'tot']]
epm_copy = epm.copy()
epm_copy['player_name'] = epm_copy['player_name'].str.lower()
epm_copy['player_name'] = epm_copy['player_name'].str.replace('.', '')
epm_recent['player_name'] = epm_recent['player_name'].apply(clean_name)
epm_recent['player_name'] = epm_recent['player_name'].str.lower()
epm_recent['player_name'] = epm_recent['player_name'].str.replace('.', '')
epm_copy['zs_epm'] = epm_copy.groupby('season')['tot'].transform(lambda x: (x - x.min()) * 10 /(x.max() - x.min()))

latest_epm = epm_recent.groupby('player_name').agg({'season': 'max'}).reset_index()
latest_epm = pd.merge(latest_epm, epm_copy[['player_name', 'season', 'tot','zs_epm']], how='left', on=['player_name', 'season'])
latest_epm = pd.merge(latest_epm, advanced_stats_df, how='left', left_on=['player_name', 'season'], right_on=['name_display', 'season'])




In [17]:

# ============================================================
# Full 2026-27 roster: value + durability, with rookie fallback for
# players with no established EPM history
# ============================================================
nba_roster_27_full = pd.merge(nba_roster_27_missed_game, latest_epm[['player_name', 'season', 'tot','zs_epm','zs_bpm','zs_ws','zs_per']], how='left', left_on='player', right_on='player_name')
nba_roster_27_full['this_season'] = 2027
nba_roster_27_full = pd.merge(nba_roster_27_full, draft[['player', 'draft_year','pick_overall']], how='left', on='player')
# nba_roster_27_full = pd.merge(nba_roster_27_full, draft[draft['draft_year'] != 2026][['player', 'draft_year']], how='left', on='player')


nba_roster_27_full['exp'] = nba_roster_27_full['this_season'] - nba_roster_27_full['draft_year']
adv_stat_weights = [
    (0.55 * nba_roster_27_full['zs_epm']) + (0.15 * nba_roster_27_full['zs_bpm']) + (0.15 * nba_roster_27_full['zs_per']) + (0.15 * nba_roster_27_full['zs_ws']),
     np.nan,
    (0.35 * nba_roster_27_full['zs_epm']) + (0.15 * nba_roster_27_full['zs_bpm']) + (0.25 * nba_roster_27_full['zs_per']) + (0.25 * nba_roster_27_full['zs_ws']),
    (0.55 * nba_roster_27_full['zs_epm']) + (0.15 * nba_roster_27_full['zs_bpm']) + (0.15 * nba_roster_27_full['zs_per']) + (0.15 * nba_roster_27_full['zs_ws'])
]
conditions = [
    (nba_roster_27_full['exp'].isna()) & (nba_roster_27_full['draft_year'].isna()),
    nba_roster_27_full['exp'] == 1,
    nba_roster_27_full['exp'] == 2,
    nba_roster_27_full['exp'] >= 3
]
nba_roster_27_full['advanced_stat_score'] = np.select(conditions,adv_stat_weights)

nba_roster_27_full.loc[(nba_roster_27_full['tot'].isna()) & (nba_roster_27_full['pick_overall'].isna()), 'pick_overall'] = 60
nba_roster_27_full = pd.merge(nba_roster_27_full, first_year_epm_by_pick, how='left', on='pick_overall', suffixes=[None, '_rookie'])
nba_roster_27_full.loc[(nba_roster_27_full['tot'].isna()), 'tot'] = nba_roster_27_full['tot_rookie']
nba_roster_27_full.loc[(nba_roster_27_full['advanced_stat_score'].isna()), 'advanced_stat_score'] = nba_roster_27_full['advanced_stat_score_rookie']
nba_roster_27_full.loc[nba_roster_27_full['team'] == 'BRK', 'team'] = 'BKN'
nba_roster_27_full.loc[nba_roster_27_full['team'] == 'CHO', 'team'] = 'CHA'
nba_roster_27_full.loc[nba_roster_27_full['team'] == 'PHO', 'team'] = 'PHX'
nba_roster_27_full.head()

,player,pos,team,pre_season_injury_flag,est_pct_season_missed,gp_pct,Team MPG Rank,player_name,season,tot,...,zs_bpm,zs_ws,zs_per,this_season,draft_year,pick_overall,exp,advanced_stat_score,tot_rookie,advanced_stat_score_rookie
0,nickeil alexander-walker,G,ATL,0,0.0,0.990000,1,nickeil alexander-walker,2026.0,2.26155,...,4.779116,4.702381,4.429066,2027,2019.0,17.0,8.0,5.201886,-2.673950,2.794960
1,jalen wilson,F,ATL,0,0.0,0.658537,17,jalen wilson,2026.0,-4.50864,...,2.048193,1.071429,1.730104,2027,2023.0,51.0,4.0,1.564224,-3.356915,2.709415
2,jock landale,C,ATL,0,0.0,0.682927,12,jock landale,2026.0,-0.42567,...,4.257028,3.214286,4.463668,2027,NaN,NaN,NaN,4.001153,NaN,NaN
3,zuby ejiofor,F,ATL,0,0.0,0.750000,13,NaN,NaN,-1.43787,...,NaN,NaN,NaN,2027,2026.0,23.0,1.0,3.554083,-1.437870,3.554083
4,henri veesaar,C,ATL,0,0.0,0.750000,18,NaN,NaN,-1.07974,...,NaN,NaN,NaN,2027,2026.0,52.0,1.0,3.333666,-1.079740,3.333666


In [18]:

# ============================================================
# Schedule x roster: every player, every game they're eligible for
# ============================================================
nba_game_roster_27 = pd.merge(schedule_27_box_df, nba_roster_27_full[['player', 'pos', 'team', 'pre_season_injury_flag', 'est_pct_season_missed', 'gp_pct', 'tot', 'advanced_stat_score', 'Team MPG Rank']], how='left', left_on='team tricode', right_on='team')
nba_game_roster_27['player'] = nba_game_roster_27['player'].str.lower()
nba_game_roster_27['player'] = nba_game_roster_27['player'].str.replace('.', '')
nba_game_roster_27.head()


,date,team tricode,opponent tricode,is_home,days_between,two_games_ago,three_games_ago,rows_ago,is_b2b,gameid,season_game_no,player,pos,team,pre_season_injury_flag,est_pct_season_missed,gp_pct,tot,advanced_stat_score,Team MPG Rank
0,2026-10-21,ATL,ORL,0,NaN,NaN,NaN,0,0,2026-10-21ATLORL,1.0,nickeil alexander-walker,G,ATL,0,0.0,0.990000,2.26155,5.201886,1
1,2026-10-21,ATL,ORL,0,NaN,NaN,NaN,0,0,2026-10-21ATLORL,1.0,jalen wilson,F,ATL,0,0.0,0.658537,-4.50864,1.564224,17
2,2026-10-21,ATL,ORL,0,NaN,NaN,NaN,0,0,2026-10-21ATLORL,1.0,jock landale,C,ATL,0,0.0,0.682927,-0.42567,4.001153,12
3,2026-10-21,ATL,ORL,0,NaN,NaN,NaN,0,0,2026-10-21ATLORL,1.0,zuby ejiofor,F,ATL,0,0.0,0.750000,-1.43787,3.554083,13
4,2026-10-21,ATL,ORL,0,NaN,NaN,NaN,0,0,2026-10-21ATLORL,1.0,henri veesaar,C,ATL,0,0.0,0.750000,-1.07974,3.333666,18


In [19]:

# ============================================================
# Historical box scores -> team-game rows with is_b2b (both sides)
# — this builds the training data for the win-probability model
# ============================================================
pd.to_datetime(full_box_score_df['gamedate'])
full_box_score_df['gamedate'] = pd.to_datetime(full_box_score_df['gamedate'], utc=True)
full_box_score_df['gamedate'] = full_box_score_df['gamedate'].dt.tz_convert('US/Pacific')
full_box_score_df['gamedate'] = full_box_score_df['gamedate'].dt.date
full_box_score_df = full_box_score_df.sort_values(by=['season', 'teamTricode', 'gamedate'], ascending=True)

season_team = pd.merge(full_box_score_df['season'].drop_duplicates(), full_box_score_df['teamTricode'].drop_duplicates(), how='cross')

new_full_box = []
for season, team in zip(season_team['season'], season_team['teamTricode']):
    temp_df = full_box_score_df[(full_box_score_df['season'] == season) & (full_box_score_df['teamTricode'] == team)]
    temp_df['days_between'] = temp_df.groupby(['season', 'teamTricode'])['gamedate'].diff()
    temp_df['two_games_ago'] = temp_df['gamedate'] - temp_df['gamedate'].shift(2)
    temp_df['three_games_ago'] = temp_df['gamedate'] - temp_df['gamedate'].shift(3)
    is_target = (temp_df["is_home"] == True).astype(int)
    temp_df["rows_ago"] = (
        is_target.groupby((is_target == 1).cumsum()).cumcount()
    )
    new_full_box.append(temp_df)

new_full_box_df = pd.concat(new_full_box)
new_full_box_df['is_b2b'] = np.where(new_full_box_df['days_between'] <= pd.to_timedelta(1, unit='D'), 1, 0)
new_full_box_df['is_home'] = new_full_box_df['is_home'].astype(int)
new_full_box_df = pd.merge(new_full_box_df, new_full_box_df[['teamTricode', 'gameid', 'is_b2b']], how='left', left_on=['gameid', 'opponent_teamTricode'], right_on=['gameid', 'teamTricode'], suffixes=[None, '_opponent'])
new_full_box_df


,Unnamed: 0,gameid,gamedate,is_home,teamId,teamTricode,teamCity,teamName,points,opponent_teamId,...,season,teamFieldGoalAttempts,sTeamFieldGoalAttempts,days_between,two_games_ago,three_games_ago,rows_ago,is_b2b,teamTricode_opponent,is_b2b_opponent
0,2437,22300063,2023-10-25,0,1610612737,ATL,Atlanta,Hawks,110,1610612766,...,2024,NaN,NaN,NaN,NaN,NaN,0,0,CHA,0
1,2420,22300079,2023-10-27,1,1610612737,ATL,Atlanta,Hawks,120,1610612752,...,2024,NaN,NaN,"2 days, 0:00:00",NaN,NaN,0,0,NYK,0
2,2389,22300097,2023-10-29,0,1610612737,ATL,Atlanta,Hawks,127,1610612749,...,2024,NaN,NaN,"2 days, 0:00:00","4 days, 0:00:00",NaN,1,0,MIL,0
3,2374,22300104,2023-10-30,1,1610612737,ATL,Atlanta,Hawks,127,1610612750,...,2024,NaN,NaN,"1 day, 0:00:00","3 days, 0:00:00","5 days, 0:00:00",0,1,MIN,0
4,2326,22300117,2023-11-01,1,1610612737,ATL,Atlanta,Hawks,130,1610612764,...,2024,NaN,NaN,"2 days, 0:00:00","3 days, 0:00:00","5 days, 0:00:00",0,0,WAS,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7375,119,22501133,2026-04-05,0,1610612764,WAS,Washington,Wizards,115,1610612751,...,2026,0.0,NaN,"1 day, 0:00:00","4 days, 0:00:00","6 days, 0:00:00",2,1,BKN,0
7376,86,22501148,2026-04-07,1,1610612764,WAS,Washington,Wizards,98,1610612741,...,2026,3.0,NaN,"2 days, 0:00:00","3 days, 0:00:00","6 days, 0:00:00",0,0,CHI,0
7377,64,22501166,2026-04-09,1,1610612764,WAS,Washington,Wizards,108,1610612741,...,2026,1.0,NaN,"2 days, 0:00:00","4 days, 0:00:00","5 days, 0:00:00",0,0,CHI,0
7378,38,22501172,2026-04-10,1,1610612764,WAS,Washington,Wizards,117,1610612748,...,2026,0.0,NaN,"1 day, 0:00:00","3 days, 0:00:00","5 days, 0:00:00",0,1,MIA,1


In [20]:

# ============================================================
# Player-level EPM merged onto historical box scores
# ============================================================
epm_filter = epm_copy.copy()
epm_filter['player_name'] = epm_filter['player_name'].apply(clean_name)
epm_filter['team_epm_rk'] = epm_filter.groupby(['team_alias', 'season'])['tot'].rank(ascending=False)
epm_filter['season'] = ('20' + epm_filter['season'].astype(str).str[-2:]).astype(int)

season_roster_epm = pd.merge(full_player_box_score_df, epm_filter[['player_name', 'season', 'tot', 'off', 'team_epm_rk','zs_epm']], how='left', left_on=['season', 'name_display'], right_on=['season', 'player_name'])
season_roster_epm['player_name'] = season_roster_epm['player_name'].str.lower()
season_roster_epm['player_name'] = season_roster_epm['player_name'].str.replace('.', '')
season_roster_epm


,Unnamed: 0,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,...,foulsPersonal,points,plusMinusPoints,season,name_display,player_name,tot,off,team_epm_rk,zs_epm
0,0,22301193,1610612763,Memphis,Grizzlies,MEM,grizzlies,1641713,GG,Jackson,...,1,44,-13.0,2024,gg jackson,gg jackson,-0.705643,-0.903329,8.0,4.172727
1,1,22301193,1610612763,Memphis,Grizzlies,MEM,grizzlies,1631222,Jake,LaRavia,...,3,29,-19.0,2024,jake laravia,jake laravia,-2.096720,-1.374090,14.0,3.358377
2,2,22301193,1610612763,Memphis,Grizzlies,MEM,grizzlies,1641998,Trey,Jemison III,...,4,6,-13.0,2024,trey jemison iii,NaN,NaN,NaN,NaN,NaN
3,3,22301193,1610612763,Memphis,Grizzlies,MEM,grizzlies,1630692,Jordan,Goodwin,...,1,8,-20.0,2024,jordan goodwin,jordan goodwin,-1.588050,-2.076380,12.0,3.656157
4,4,22301193,1610612763,Memphis,Grizzlies,MEM,grizzlies,1630590,Scotty,Pippen Jr.,...,5,15,-1.0,2024,scotty pippen jr,scotty pippen jr,0.756708,-0.277290,4.0,5.028801
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97074,32174,22500002,1610612744,Golden State,Warriors,GSW,warriors,1630611,Gui,Santos,...,0,0,0.0,2026,gui santos,gui santos,-0.262515,-0.185462,10.0,4.119666
97075,32175,22500002,1610612744,Golden State,Warriors,GSW,warriors,1642954,Will,Richard,...,2,5,-11.0,2026,will richard,will richard,-1.729880,-1.710270,15.0,3.221760
97076,32176,22500002,1610612744,Golden State,Warriors,GSW,warriors,1642366,Quinten,Post,...,1,0,5.0,2026,quinten post,quinten post,0.008904,-0.139268,8.0,4.285752
97077,32177,22500002,1610612744,Golden State,Warriors,GSW,warriors,1631218,Trayce,Jackson-Davis,...,0,0,0.0,2026,trayce jackson-davis,trayce jackson-davis,0.338520,-1.851280,7.0,4.487449


In [21]:

# ============================================================
# Build the top-8-by-minutes team EPM/advanced-stat differential
# per historical game — the actual model training features
# ============================================================
new_full_box_df['season'] = new_full_box_df['season'].astype(int)
box_score_epm = pd.merge(new_full_box_df, season_roster_epm, how='left', left_on=['gameid', 'teamTricode'], right_on=['gameId', 'teamTricode'])
box_score_epm['minutes_y'] = pd.to_timedelta("00:" + box_score_epm['minutes_y']).dt.total_seconds() / 60
box_score_epm['game_team_min_rk'] = box_score_epm.groupby(['gameid', 'teamTricode'])['minutes_y'].rank(ascending=False)
box_score_epm = box_score_epm[box_score_epm['game_team_min_rk'] <= 8]
box_score_epm = pd.merge(box_score_epm, advanced_stats_df, how='left', left_on=['player_name', 'season_x'], right_on=['name_display', 'season'])
box_score_epm = pd.merge(box_score_epm, draft[['player', 'draft_year']], how='left', left_on='player_name',right_on='player')


box_score_epm['exp'] = box_score_epm['season_x'] - box_score_epm['draft_year']
adv_stat_weights = [
    (0.55 * box_score_epm['zs_epm']) + (0.15 * box_score_epm['zs_bpm']) + (0.15 * box_score_epm['zs_per']) + (0.15 * box_score_epm['zs_ws']),
     np.nan,
    (0.35 * box_score_epm['zs_epm']) + (0.15 * box_score_epm['zs_bpm']) + (0.25 * box_score_epm['zs_per']) + (0.25 * box_score_epm['zs_ws']),
    (0.55 * box_score_epm['zs_epm']) + (0.15 * box_score_epm['zs_bpm']) + (0.15 * box_score_epm['zs_per']) + (0.15 * box_score_epm['zs_ws'])
]
conditions = [
    (box_score_epm['exp'].isna()) & (box_score_epm['draft_year'].isna()),
    box_score_epm['exp'] == 1,
    box_score_epm['exp'] == 2,
    box_score_epm['exp'] >= 3
]
box_score_epm['advanced_stat_score'] = np.select(conditions,adv_stat_weights)

temp_team_box_score_epm = box_score_epm.groupby(['gameid', 'teamTricode']).agg({'tot': 'sum', 'off': 'sum', 'advanced_stat_score': 'sum'}).reset_index()

full_box_temp = pd.merge(new_full_box_df, temp_team_box_score_epm, how='left', on=['gameid', 'teamTricode'])
full_box_epm = pd.merge(full_box_temp, temp_team_box_score_epm, how='left', left_on=['gameid', 'opponent_teamTricode'], right_on=['gameid', 'teamTricode'], suffixes=[None, '_opponent'])
full_box_epm['top_8_diff'] = full_box_epm['tot'] - full_box_epm['tot_opponent']
full_box_epm['top_8_off_diff'] = full_box_epm['off'] - full_box_epm['off_opponent']
full_box_epm['top_8_adv_diff'] = full_box_epm['advanced_stat_score'] - full_box_epm['advanced_stat_score_opponent']
box_score_epm

,Unnamed: 0_x,gameid,gamedate,is_home,teamId_x,teamTricode,teamCity_x,teamName_x,points_x,opponent_teamId,...,per,ws,bpm,zs_bpm,zs_per,zs_ws,player,draft_year,exp,advanced_stat_score
0,2437,22300063,2023-10-25,0,1610612737,ATL,Atlanta,Hawks,110,1610612766,...,12.7,2.2,-2.4,3.361702,3.096774,2.085561,de'andre hunter,2019.0,5.0,3.471282
1,2437,22300063,2023-10-25,0,1610612737,ATL,Atlanta,Hawks,110,1610612766,...,13.1,3.5,-1.8,3.617021,3.225806,2.780749,saddiq bey,2020.0,4.0,3.725006
2,2437,22300063,2023-10-25,0,1610612737,ATL,Atlanta,Hawks,110,1610612766,...,20.7,6.3,0.1,4.425532,5.677419,4.278075,clint capela,2014.0,10.0,5.071088
3,2437,22300063,2023-10-25,0,1610612737,ATL,Atlanta,Hawks,110,1610612766,...,17.7,4.9,1.7,5.106383,4.709677,3.529412,dejounte murray,2016.0,8.0,4.756568
4,2437,22300063,2023-10-25,0,1610612737,ATL,Atlanta,Hawks,110,1610612766,...,20.3,4.6,2.6,5.489362,5.548387,3.368984,trae young,2018.0,6.0,5.325991
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62502,29,22501187,2026-04-12,0,1610612764,WAS,Washington,Wizards,117,1610612739,...,12.1,0.4,-4.5,2.489960,2.906574,1.190476,will riley,2025.0,1.0,NaN
62503,29,22501187,2026-04-12,0,1610612764,WAS,Washington,Wizards,117,1610612739,...,9.9,-0.4,-4.7,2.409639,2.145329,0.714286,bub carrington,2024.0,2.0,1.579854
62504,29,22501187,2026-04-12,0,1610612764,WAS,Washington,Wizards,117,1610612739,...,13.8,0.6,-3.9,2.730924,3.494810,1.309524,sharife cooper,2021.0,5.0,2.352734
62505,29,22501187,2026-04-12,0,1610612764,WAS,Washington,Wizards,117,1610612739,...,10.9,-0.2,-4.8,2.369478,2.491349,0.833333,jaden hardy,2022.0,4.0,2.514441


In [22]:
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
# ============================================================
# WIN PROBABILITY MODEL — this is the 'result' object the simulation uses
# ============================================================
full_box_epm['team_season'] = full_box_epm['season'].astype(str) + '_' + full_box_epm['teamTricode']
X = full_box_epm[['is_home', 'is_b2b_opponent', 'is_b2b', 'top_8_adv_diff']]
y = full_box_epm['win']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize and fit the model (lbfgs is the default solver)
model = LogisticRegression(max_iter=10000)  # Increased max_iter helps the solver converge
model.fit(X_train, y_train)

# 4. Make predictions
y_pred = model.predict(X_test)         # Predicts hard labels (0 or 1)
y_probs = model.predict_proba(X_test)  # Predicts continuous probabilities

# 5. Evaluate the model
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Add a constant to include an intercept in the model
X = sm.add_constant(X)
# Fit model
model = sm.Logit(y, X)
result = model.fit( cov_type='cluster',
    cov_kwds={'groups': full_box_epm['team_season']})

# View statistics
print(result.summary())


Accuracy: 0.7026

Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.67      0.69       722
           1       0.70      0.73      0.72       754

    accuracy                           0.70      1476
   macro avg       0.70      0.70      0.70      1476
weighted avg       0.70      0.70      0.70      1476

Optimization terminated successfully.
         Current function value: 0.572939
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:                    win   No. Observations:                 7380
Model:                          Logit   Df Residuals:                     7375
Method:                           MLE   Df Model:                            4
Date:                Sun, 04 Oct 2026   Pseudo R-squ.:                  0.1734
Time:                        09:54:25   Log-Likelihood:                -4228.3
converged:                       True   LL-Null:       

In [23]:
# # ============================================================
# # SEASON SIMULATION
# # ============================================================
# rng = np.random.default_rng()
#
# sims = []
# for i in range(1, 10001):
#     print(f"\rsim {i}/{10000}", end="", flush=True)
#     sim = []
#     for team in schedule_27_box_df['team tricode'].drop_duplicates():
#         team_games = nba_game_roster_27[nba_game_roster_27['team tricode'] == team]
#
#         team_players_gp = []
#         for player in team_games['player'].dropna().drop_duplicates():
#             game_player = team_games[team_games['player'] == player][['player', 'team', 'est_pct_season_missed', 'gp_pct']].drop_duplicates()
#             if game_player[['est_pct_season_missed']].drop_duplicates().iat[0, 0] > 0:
#                 player_games_played = rng.binomial(1, (1 - game_player['est_pct_season_missed'].drop_duplicates()), 82)
#                 player_missed_games = (82 - sum(player_games_played)).round(0)
#                 injury_start = 1
#             else:
#                 player_games_played = rng.binomial(1, game_player['gp_pct'].drop_duplicates(), 82)
#                 player_missed_games = (82 - sum(player_games_played)).round(0)
#                 if player_missed_games >= 82:
#                     injury_start = 1
#                     player_missed_games = 82
#                 elif player_missed_games == 0:
#                     injury_start = 0
#                     player_missed_games = 0
#                 else:
#                     injury_start = rng.integers(1, 82 - player_missed_games + 2)
#             missed_game_indices = range(injury_start, injury_start + player_missed_games)
#             team_players_gp.append([team, player, player_missed_games, missed_game_indices])
#
#         team_player_gp = pd.DataFrame(team_players_gp, columns=['team', 'player', 'player_missed_games', 'missed_game_no'])
#         team_player_gp = team_player_gp.explode('missed_game_no', ignore_index=True)
#         team_games = pd.merge(team_games, team_player_gp, how='left', left_on=['player', 'season_game_no'], right_on=['player', 'missed_game_no'])
#         team_games['is_playing'] = np.where(team_games['missed_game_no'].isna(), 1, 0)
#         team_games['sim_no'] = i
#         sim.append(team_games)
#
#     sim_season_game_roster = pd.concat(sim)
#     sim_season_game_roster_sorted = sim_season_game_roster.copy()
#
#     sim_season_game_roster_sorted['sort_key'] = np.where(
#         sim_season_game_roster_sorted['is_playing'] == 1,
#         sim_season_game_roster_sorted['Team MPG Rank'],
#         -sim_season_game_roster_sorted['Team MPG Rank']
#     )
#
#     sim_season_game_roster_sorted = sim_season_game_roster_sorted.sort_values(
#         ['date', 'team tricode', 'is_playing', 'sort_key'],
#         ascending=[True, True, False, True]
#     )
#
#     sim_season_game_roster_sorted['adj_team_mpg_rank'] = (
#         sim_season_game_roster_sorted.groupby(['date', 'team tricode']).cumcount() + 1
#     )
#
#     sim_season_game = sim_season_game_roster_sorted[sim_season_game_roster_sorted['adj_team_mpg_rank'] <= 8].groupby(
#         ['date', 'team tricode', 'opponent tricode', 'is_home', 'is_b2b', 'season_game_no', 'sim_no']
#     ).agg({'tot': 'sum', 'advanced_stat_score': 'sum'}).reset_index()
#
#     sim_season_game = pd.merge(sim_season_game, sim_season_game[['date', 'team tricode', 'is_b2b', 'tot', 'advanced_stat_score']], how='left', left_on=['date', 'opponent tricode'], right_on=['date', 'team tricode'], suffixes=[None, '_opponent'])
#     sim_season_game['game_epm_diff'] = sim_season_game['tot'] - sim_season_game['tot_opponent']
#     sim_season_game['game_adv_diff'] = sim_season_game['advanced_stat_score'] - sim_season_game['advanced_stat_score_opponent']
#     sim_season_game['w_prob'] = result.predict(sm.add_constant(sim_season_game[['is_home', 'is_b2b_opponent', 'is_b2b', 'game_adv_diff']]))
#     sim_season_game = sim_season_game[sim_season_game['is_home'] == 1]
#     sim_season_game['win'] = rng.binomial(1, sim_season_game['w_prob'])
#
#     home_wins = sim_season_game.groupby(['team tricode', 'sim_no']).agg(home_wins=('win', 'sum')).reset_index()
#     road_wins = sim_season_game.groupby(['opponent tricode', 'sim_no']).agg(road_wins=('win', 'sum')).reset_index()
#     road_wins['road_wins'] = 41 - road_wins['road_wins']
#
#     league_records = pd.merge(home_wins, road_wins, how='left', left_on=['team tricode', 'sim_no'], right_on=['opponent tricode', 'sim_no'])
#     league_records = league_records.drop(columns='opponent tricode')
#     league_records['wins'] = league_records['home_wins'] + league_records['road_wins']
#     league_records['losses'] = 82 - league_records['wins']
#     league_records['w/l'] = league_records['wins'] / 82
#     league_records['league_draft_rk'] = league_records['w/l'].rank(ascending=True, method="first")
#     league_records = league_records[['team tricode','w/l','league_draft_rk']]
#     league_records = league_records.sort_values(by='league_draft_rk').reset_index(drop=True)
#     sim_season_lottery_balls = pd.merge(league_records,lottery_balls,how='left',left_on='league_draft_rk',right_on='Worst Record').reset_index()
#     lottery_results = []
#     for j,draw_no in enumerate(sorted(league_records[league_records['league_draft_rk'] <= 16]['league_draft_rk'])):
#         if (draw_no == 10) & (len(sim_season_lottery_balls[sim_season_lottery_balls['league_draft_rk'] <= 3]['team tricode'].drop_duplicates()) == 3):
#             selected = rng.choice(sim_season_lottery_balls[sim_season_lottery_balls['league_draft_rk'] <= 3].index)
#         elif (draw_no == 11) & (len(sim_season_lottery_balls[sim_season_lottery_balls['league_draft_rk'] <= 3]['team tricode'].drop_duplicates()) == 2):
#             selected = rng.choice(sim_season_lottery_balls[sim_season_lottery_balls['league_draft_rk'] <= 3].index)
#         elif (draw_no == 12) & (len(sim_season_lottery_balls[sim_season_lottery_balls['league_draft_rk'] <= 3]['team tricode'].drop_duplicates()) == 1):
#             selected = sim_season_lottery_balls['league_draft_rk'].idxmin()
#         else:
#             if draw_no == 1:
#                 selected = rng.choice(sim_season_lottery_balls[(sim_season_lottery_balls['league_draft_rk'] <= 16) & (~sim_season_lottery_balls['team tricode'].isin(first_pick_26)) & (~sim_season_lottery_balls['team tricode'].isin(top_five_25_26))].index)
#             elif draw_no <= 5:
#                 selected = rng.choice(sim_season_lottery_balls[(sim_season_lottery_balls['league_draft_rk'] <= 16)  & (~sim_season_lottery_balls['team tricode'].isin(top_five_25_26))].index)
#             else:
#                 selected = rng.choice(sim_season_lottery_balls[(sim_season_lottery_balls['league_draft_rk'] <= 16)].index)
#         selected_team = sim_season_lottery_balls[sim_season_lottery_balls.index == selected]['team tricode'][selected]
#         lottery_results.append([draw_no, sim_season_lottery_balls[sim_season_lottery_balls.index == selected]['team tricode'][selected]])
#         sim_season_lottery_balls = sim_season_lottery_balls[sim_season_lottery_balls['team tricode'] != selected_team]
#     lottery_results = pd.DataFrame(lottery_results,columns=['pick_no','team tricode'])
#     draft_order = pd.concat([lottery_results,league_records[league_records['league_draft_rk'] > 16][['league_draft_rk','team tricode']].rename(columns={'league_draft_rk':'pick_no'}).sort_values(by='pick_no')])
#     league_records_27 = pd.merge(league_records,draft_order,how='left',on='team tricode')
#     league_records_27['season'] = 2027
#
#     #2028
#     league_records_28 = league_records_27.copy()
#     league_records_28['season'] = 2028
#     league_records_28['w/l'] = league_records_28['w/l'] + (0.1986  +  -0.3970  * league_records_28['w/l'])
#     league_records_28 = league_records_28[['team tricode','w/l','league_draft_rk','season']]
#
#     first_pick_27 = league_records_27[ (league_records_27['pick_no'] == 1)]['team tricode']
#     top_five_26_27 = pd.merge(nba_draft_history[(nba_draft_history['draft_year'] == 2026) & (nba_draft_history['pick_overall'] <= 5)],league_records_27[ (league_records_27['pick_no'] <= 5)], how='inner',left_on='team_id',right_on = 'team tricode')['team tricode']
#
#     sim_28_df = league_records_28.copy()
#
#     sim_28_df['sim_w/l'] = rng.binomial(82,sim_28_df['w/l']) / 82
#
#     sim_28_df['league_draft_rk'] = sim_28_df['sim_w/l'].rank(ascending=True, method="first")
#     sim_season_lottery_balls = pd.merge(sim_28_df,lottery_balls,how='left',left_on='league_draft_rk',right_on='Worst Record').reset_index()
#     lottery_results = []
#     for j,draw_no in enumerate(sorted(sim_28_df[sim_28_df['league_draft_rk'] <= 16]['league_draft_rk'])):
#         if (draw_no == 10) & (len(sim_season_lottery_balls[sim_season_lottery_balls['league_draft_rk'] <= 3]['team tricode'].drop_duplicates()) == 3):
#             selected = rng.choice(sim_season_lottery_balls[sim_season_lottery_balls['league_draft_rk'] <= 3].index)
#         elif (draw_no == 11) & (len(sim_season_lottery_balls[sim_season_lottery_balls['league_draft_rk'] <= 3]['team tricode'].drop_duplicates()) == 2):
#             selected = rng.choice(sim_season_lottery_balls[sim_season_lottery_balls['league_draft_rk'] <= 3].index)
#         elif (draw_no == 12) & (len(sim_season_lottery_balls[sim_season_lottery_balls['league_draft_rk'] <= 3]['team tricode'].drop_duplicates()) == 1):
#             selected = sim_season_lottery_balls['league_draft_rk'].idxmin()
#         else:
#             if draw_no == 1:
#                 selected = rng.choice(sim_season_lottery_balls[(sim_season_lottery_balls['league_draft_rk'] <= 16) & (~sim_season_lottery_balls['team tricode'].isin(first_pick_27)) & (~sim_season_lottery_balls['team tricode'].isin(top_five_26_27))].index)
#             elif draw_no <= 5:
#                 selected = rng.choice(sim_season_lottery_balls[(sim_season_lottery_balls['league_draft_rk'] <= 16)  & (~sim_season_lottery_balls['team tricode'].isin(top_five_26_27))].index)
#             else:
#                 selected = rng.choice(sim_season_lottery_balls[(sim_season_lottery_balls['league_draft_rk'] <= 16)].index)
#         selected_team = sim_season_lottery_balls[sim_season_lottery_balls.index == selected]['team tricode'][selected]
#         lottery_results.append([draw_no, sim_season_lottery_balls[sim_season_lottery_balls.index == selected]['team tricode'][selected]])
#         sim_season_lottery_balls = sim_season_lottery_balls[sim_season_lottery_balls['team tricode'] != selected_team]
#     lottery_results = pd.DataFrame(lottery_results,columns=['pick_no','team tricode'])
#     draft_order = pd.concat([lottery_results,sim_28_df[sim_28_df['league_draft_rk'] > 16][['league_draft_rk','team tricode']].rename(columns={'league_draft_rk':'pick_no'}).sort_values(by='pick_no')])
#     sim_28_df = pd.merge(sim_28_df,draft_order,how='left',on='team tricode')
#     sim_28_df['w/l'] = sim_28_df['sim_w/l']
#     sim_28_df = sim_28_df.drop(columns='sim_w/l')
#
#     #2029
#     league_records_29 = sim_28_df.copy()
#     league_records_29['season'] = 2029
#     league_records_29['w/l'] = league_records_29['w/l'] + (0.1986  +  -0.3970  * league_records_29['w/l'])
#     league_records_29 = league_records_29[['team tricode','w/l','league_draft_rk','season']]
#
#     first_pick_28 = sim_28_df[ (sim_28_df['pick_no'] == 1)]['team tricode']
#     top_five_27_28 = pd.merge(sim_28_df[ (sim_28_df['pick_no'] <= 5)],league_records_27[ (league_records_27['pick_no'] <= 5)], how='inner',left_on='team tricode',right_on = 'team tricode')['team tricode']
#
#     sim_29_df = league_records_29.copy()
#
#     sim_29_df['sim_w/l'] = rng.binomial(82,sim_29_df['w/l']) / 82
#
#     sim_29_df['league_draft_rk'] = sim_29_df['sim_w/l'].rank(ascending=True, method="first")
#     sim_season_lottery_balls = pd.merge(sim_29_df,lottery_balls,how='left',left_on='league_draft_rk',right_on='Worst Record').reset_index()
#     lottery_results = []
#     for j,draw_no in enumerate(sorted(sim_29_df[sim_29_df['league_draft_rk'] <= 16]['league_draft_rk'])):
#         if (draw_no == 10) & (len(sim_season_lottery_balls[sim_season_lottery_balls['league_draft_rk'] <= 3]['team tricode'].drop_duplicates()) == 3):
#             selected = rng.choice(sim_season_lottery_balls[sim_season_lottery_balls['league_draft_rk'] <= 3].index)
#         elif (draw_no == 11) & (len(sim_season_lottery_balls[sim_season_lottery_balls['league_draft_rk'] <= 3]['team tricode'].drop_duplicates()) == 2):
#             selected = rng.choice(sim_season_lottery_balls[sim_season_lottery_balls['league_draft_rk'] <= 3].index)
#         elif (draw_no == 12) & (len(sim_season_lottery_balls[sim_season_lottery_balls['league_draft_rk'] <= 3]['team tricode'].drop_duplicates()) == 1):
#             selected = sim_season_lottery_balls['league_draft_rk'].idxmin()
#         else:
#             if draw_no == 1:
#                 selected = rng.choice(sim_season_lottery_balls[(sim_season_lottery_balls['league_draft_rk'] <= 16) & (~sim_season_lottery_balls['team tricode'].isin(first_pick_28)) & (~sim_season_lottery_balls['team tricode'].isin(top_five_27_28))].index)
#             elif draw_no <= 5:
#                 selected = rng.choice(sim_season_lottery_balls[(sim_season_lottery_balls['league_draft_rk'] <= 16)  & (~sim_season_lottery_balls['team tricode'].isin(top_five_27_28))].index)
#             else:
#                 selected = rng.choice(sim_season_lottery_balls[(sim_season_lottery_balls['league_draft_rk'] <= 16)].index)
#         selected_team = sim_season_lottery_balls[sim_season_lottery_balls.index == selected]['team tricode'][selected]
#         lottery_results.append([draw_no, sim_season_lottery_balls[sim_season_lottery_balls.index == selected]['team tricode'][selected]])
#         sim_season_lottery_balls = sim_season_lottery_balls[sim_season_lottery_balls['team tricode'] != selected_team]
#     lottery_results = pd.DataFrame(lottery_results,columns=['pick_no','team tricode'])
#     draft_order = pd.concat([lottery_results,sim_29_df[sim_29_df['league_draft_rk'] > 16][['league_draft_rk','team tricode']].rename(columns={'league_draft_rk':'pick_no'}).sort_values(by='pick_no')])
#     sim_29_df = pd.merge(sim_29_df,draft_order,how='left',on='team tricode')
#     sim_29_df['w/l'] = sim_29_df['sim_w/l']
#     sim_29_df = sim_29_df.drop(columns='sim_w/l')
#
#     final_sim_df = pd.concat([league_records_27,sim_28_df,sim_29_df]).reset_index(drop=True)
#     rd_2_27_1 = final_sim_df[(final_sim_df['season'] == 2027) & (final_sim_df['pick_no'] <= 16)]
#     rd_2_27_1['pick_no'] = rd_2_27_1['pick_no'].rank(ascending=False) + 30
#     rd_2_27_2 = final_sim_df[(final_sim_df['season'] == 2027) & (final_sim_df['pick_no'] > 16)]
#     rd_2_27_2['pick_no'] = rd_2_27_2['w/l'].rank(ascending=True,method="first") + 30 + 16
#
#     rd_2_28_1 = final_sim_df[(final_sim_df['season'] == 2028) & (final_sim_df['pick_no'] <= 16)]
#     rd_2_28_1['pick_no'] = rd_2_28_1['pick_no'].rank(ascending=False) + 30
#     rd_2_28_2 = final_sim_df[(final_sim_df['season'] == 2028) & (final_sim_df['pick_no'] > 16)]
#     rd_2_28_2['pick_no'] = rd_2_28_2['w/l'].rank(ascending=True,method="first") + 30 + 16
#
#     rd_2_29_1 = final_sim_df[(final_sim_df['season'] == 2029) & (final_sim_df['pick_no'] <= 16)]
#     rd_2_29_1['pick_no'] = rd_2_29_1['pick_no'].rank(ascending=False) + 30
#     rd_2_29_2 = final_sim_df[(final_sim_df['season'] == 2029) & (final_sim_df['pick_no'] > 16)]
#     rd_2_29_2['pick_no'] = rd_2_29_2['w/l'].rank(ascending=True,method="first") + 30 + 16
#
#     final_full_draft_results = pd.concat([final_sim_df,rd_2_27_1,rd_2_27_2,rd_2_28_1,rd_2_28_2,rd_2_29_1,rd_2_29_2])
#
#     # #CLIPPERS PICK DOCKED
#     # final_full_draft_results.loc[(final_full_draft_results['season'] == 2029) & (final_full_draft_results['pick_no'] <= 30) & (final_full_draft_results['team tricode'] != 'LAC') & (final_full_draft_results['pick_no'] >
#     #                                                                                                                                                                                  (final_full_draft_results.loc[(final_full_draft_results['season'] == 2029) & (final_full_draft_results['pick_no'] <= 30) & (final_full_draft_results['team tricode'] == 'LAC'),'pick_no'].iloc[0])),'pick_no' ] -= 1
#     # final_full_draft_results = final_full_draft_results[~((final_full_draft_results['season'] == 2029) & (final_full_draft_results['pick_no'] <= 30) & (final_full_draft_results['team tricode'] == 'LAC'))]
#
#     final_full_draft_results['sim_no'] = i
#     final_full_draft_results = pd.merge(final_full_draft_results,draft_pick_value[['pick_overall','value']],how='left',left_on='pick_no',right_on='pick_overall')
#     sims.append(final_full_draft_results)
#
# all_sim_df = pd.concat(sims)
# all_sim_df['round'] = np.where(all_sim_df['pick_no'] <= 30, 1, 2)
# combo_counts = all_sim_df.groupby(['team tricode', 'season', 'pick_no', 'round', 'value']).size().reset_index(name='count')
# most_common_pick_value = combo_counts[['team tricode', 'season','round', 'pick_no', 'value','count']]
# most_common_pick_value['weight'] = most_common_pick_value['count'] / len(sims)
# most_common_pick_value['weighted_value'] = most_common_pick_value['value'] * most_common_pick_value['weight']
# mean_simmed_record = most_common_pick_value.groupby(['team tricode','season','round']).agg({'weighted_value':'sum'}).reset_index()
# # mean_simmed_record['league_rk'] = mean_simmed_record.groupby('season')['w/l'].rank(ascending=False, method="first")
# # mean_simmed_record['league_draft_rk'] = mean_simmed_record.groupby('season')['w/l'].rank(ascending=True, method="first")
# mean_simmed_record

In [24]:
# mean_simmed_record.to_csv('NBA_season_sim_2027-29.csv')
mean_simmed_record = pd.read_csv('NBA_season_sim_2027-29.csv')
mean_simmed_record

,Unnamed: 0,team tricode,season,round,weighted_value
0,0,ATL,2027,1,3.428453
1,1,ATL,2027,2,3.117652
2,2,ATL,2028,1,3.429438
3,3,ATL,2028,2,3.111419
4,4,ATL,2029,1,3.432408
...,...,...,...,...,...
175,175,WAS,2027,2,3.146530
176,176,WAS,2028,1,3.551739
177,177,WAS,2028,2,3.146346
178,178,WAS,2029,1,3.548968


In [25]:
owned_draft_pick_value = pd.merge(team_draft_picks,mean_simmed_record,how='left',right_on=['team tricode','season','round'],left_on=['originating_team','year','round'])
owned_draft_pick_value.loc[(owned_draft_pick_value['year'] >= 2030) & (owned_draft_pick_value['round'] == 1),'weighted_value'] = 3.382987
owned_draft_pick_value.loc[(owned_draft_pick_value['year'] >= 2030) & (owned_draft_pick_value['round'] == 2),'weighted_value'] = 3.108334
owned_draft_pick_value.loc[(owned_draft_pick_value['originating_team'] == 'IND') & (owned_draft_pick_value['round'] == 1) & (owned_draft_pick_value['rights_holder'] == 'LAC'),'notes'] = 'In question RE: Salary Cap Circumvention'
owned_draft_pick_value.loc[(owned_draft_pick_value['year'] >= 2030) & (owned_draft_pick_value['originating_team'] == 'LAC') & (owned_draft_pick_value['round'] == 1),'notes'] = 'In question RE: Salary Cap Circumvention'
owned_draft_pick_value = owned_draft_pick_value.dropna(subset='weighted_value')
owned_draft_pick_value = owned_draft_pick_value[[ 'originating_team',
       'year', 'round', 'rights_holder', 'notes','weighted_value']]
owned_draft_pick_value.loc[owned_draft_pick_value['notes'].isna(), 'notes'] = '-'
owned_draft_pick_value.rename(columns={'weighted_value':'value'},inplace=True)
owned_draft_pick_value['value'] = owned_draft_pick_value['value'].round(2)
owned_draft_pick_value['% of league value'] = (owned_draft_pick_value['value'] * 1.000 / owned_draft_pick_value['value'].sum()) * 100
owned_draft_pick_value = owned_draft_pick_value.rename(columns={'originating_team':'originating team','rights_holder':'rights holder'})
owned_draft_pick_value.to_csv('NBA_modeled_draft_pick_value.csv')
owned_draft_pick_value

,originating team,year,round,rights holder,notes,value,% of league value
0,ATL,2027,1,SAS,No protections,3.43,0.248441
1,ATL,2027,2,ATL,-,3.12,0.225987
2,ATL,2028,1,ATL,-,3.43,0.248441
3,ATL,2028,2,BKN,No protections,3.11,0.225263
4,ATL,2029,1,ATL,-,3.43,0.248441
...,...,...,...,...,...,...,...
419,WAS,2031,2,LAL,No protections,3.11,0.225263
420,WAS,2032,1,WAS,-,3.38,0.244819
421,WAS,2032,2,LAL,No protections,3.11,0.225263
422,WAS,2033,1,WAS,-,3.38,0.244819


In [26]:
# LOAD DATA TO SUPABASE
import os
from dotenv import load_dotenv
from supabase import create_client, Client

# Load environment variables from .env file
load_dotenv()

url: str = os.environ.get("SUPABASE_URL")
key: str = os.environ.get("SUPABASE_KEY")

# Initialize the client
supabase: Client = create_client(url, key)

records = owned_draft_pick_value.to_dict(orient="records")

# Bulk load to Supabase
response = (
    supabase.table("nba_draft_pick_value")
    .delete()
    .neq("originating team", -1)
    .execute()
)

response = supabase.table("nba_draft_pick_value").insert(records).execute()

In [27]:
owned_draft_pick_value.columns

Index(['originating team', 'year', 'round', 'rights holder', 'notes', 'value',
       '% of league value'],
      dtype='object')

In [28]:
from dash import Dash, html, dcc, callback, Output, Input
import dash_bootstrap_components as dbc
import dash_ag_grid as dag
import pandas as pd

app = Dash(external_stylesheets=[dbc.themes.BOOTSTRAP,
    "https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;600&display=swap"])

base_grid_options = {
    "theme": {"function": "themeAlpine.withParams({backgroundColor: '#000000', foregroundColor: '#fff', headerBackgroundColor: '#000000', fontFamily: 'IBM Plex Mono, monospace'})"},
    "animateRows": True,
    "pinnedBottomRowData": [],
    "context": {"valueMin": 0, "valueMax": 1},
}

# columnDefs are now STATIC — cellStyle reads min/max from context at runtime, not baked in
column_defs = [
    {"field": "originating team"},
    {"field": "year"},
    {"field": "round"},
    {"field": "rights holder"},
    {"field": "notes"},
    {
        "field": "value",
        "valueFormatter": {"function": "d3.format(',.2f')(params.value)"},
        "cellStyle": {"function": "valueGradient(params)"},
    },
    {"field": "% of league value", "valueFormatter": {"function": "d3.format(',.2f')(params.value)"}},
]

app.layout = html.Div([html.Div([
    #ENTIRE INNER BOX
    html.Div([

    html.Div([
        html.H1("NBA Draft Pick Value", style={
            "backgroundColor": "#000000",
            "color": "#fff",
            "padding": "15px",
            "borderRadius": "6px",
            "fontFamily": "Barlow Condensed, sans-serif",
            "fontWeight":"500",
            "fontSize": "40px",
            "margin": "0",
        }),
        html.P(
            "Value is based on an adjusted composite of player value metrics over their first four seasons set on a scale from 0-10, 10 being equivalent to the value of a player who ranked first in the league in all advanced stat categories in a season. The value of the 1st overall pick is 4.06, roughly equivalent to that of players such as Kon Knueppel, De'Aaron Fox, Zach Edey, and Lauri Markkanen. All picks beyond 2029 are valued at the middle of each round.",
            style={
                "color": "#ccc",
                "fontSize": "14px",
                "margin": "0",
                "maxWidth": "450px",
                "marginTop": "15px",
                "marginRight": "80px",
                "textAlign": "left",
            }
        ),
    ], style={
        "display": "flex",
        "width": "100%",
        "justifyContent": "space-between",
        "alignItems": "center",
        "backgroundColor": "#000000",
        "padding": "15px",
        "borderRadius": "6px",
    }),
#DROPDOWN CONTAINER
    dbc.Container([
        dbc.Row([
            dbc.Col(dcc.Dropdown(
                owned_draft_pick_value['originating team'].unique(),
                id='origin-dropdown', value=[], multi=True, closeOnSelect=True,
                style={'height': '30px', 'width': '200px','color':'black'}, placeholder="Originating Team",
            ), width="auto"),
            dbc.Col(dcc.Dropdown(
                owned_draft_pick_value['rights holder'].unique(),
                id='rights-dropdown', value=[], multi=True, closeOnSelect=True,
                style={'height': '30px', 'width': '200px','color':'black'}, placeholder="Rights Holder",
            ), width="auto"),
        ])
    ], fluid=True, style={
        "backgroundColor": "#000000",
        "color": "#fff",
        "fontFamily": "Barlow Condensed, sans-serif",
        "padding": "15px",
        "borderRadius": "6px",
        "border": "1px solid #000000"
    }),

    dag.AgGrid(
        id='my-datatable',
        rowData=owned_draft_pick_value.to_dict('records'),
        columnDefs=column_defs,
        defaultColDef={"sortable": True, "filter": True, "resizable": True, "flex" : 1},
        style={"flex": "1", "width": "100%",  "borderRadius": "3px",
            "overflow": "hidden",
            "border": "1px solid #3E424B"},
        dashGridOptions=base_grid_options,
        dangerously_allow_code=True,
    ),

], style={
    "backgroundColor": "#000000",
    "color": "#000000",
    "display": "flex",
    "flexDirection": "column",
    "height": "100vh",
    'margin': '0 auto',
    "borderRadius": "6px",
     "width": "98%",
     "border": "0.2px solid #000000"
})], style={
    "backgroundColor": "#000000",
    "color": "#000000",
    "display": "flex",
    "flexDirection": "column",
    "height": "100vh",
    "borderRadius": "3px",
    'margin': '0 auto',
    "width": "99%",
    "height": "99%",
    "overflow": "hidden",
    "border": "0.2px solid #000000"})], style={
    "backgroundColor": "#000000",
    "color": "#000000",
    "display": "flex",
    "flexDirection": "column",
    "height": "100vh",
    'margin': '0 auto',
    "overflow": "hidden"})


@callback(
    Output('my-datatable', 'rowData'),
    Input('origin-dropdown', 'value'),
    Input('rights-dropdown', 'value')
)
def update_output(teams, rights_holders):
    filtered_df = owned_draft_pick_value
    if teams:
        filtered_df = filtered_df[filtered_df['originating team'].isin(teams)]
    if rights_holders:
        filtered_df = filtered_df[filtered_df['rights holder'].isin(rights_holders)]
    return filtered_df.to_dict('records')


# Single callback now handles BOTH the pinned total row AND the gradient's min/max context,
# since both depend on whatever rows are currently visible (virtualRowData).
@callback(
    Output('my-datatable', 'dashGridOptions'),
    Input('my-datatable', 'virtualRowData')
)
def update_grid_options(virtual_data):
    total_value = 0
    total_pct_value = 0
    value_min = 0
    value_max = 1

    if virtual_data:
        df_current = pd.DataFrame(virtual_data)
        total_value = round(df_current["value"].sum(), 2)
        total_pct_value = round(df_current["% of league value"].sum(), 2)
        value_min = float(df_current["value"].min())
        value_max = float(df_current["value"].max())

    total_row = [{
        'originating team': 'Total',
        'year': 0,
        'round': 0,
        'rights holder': '',
        'notes': '',
        'value': total_value,
        '% of league value': total_pct_value
    }]

    updated_options = dict(base_grid_options)
    updated_options["pinnedBottomRowData"] = total_row
    updated_options["context"] = {"valueMin": value_min, "valueMax": value_max}
    return updated_options


if __name__ == '__main__':
    app.run(debug=True, port=8050)

In [29]:
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split


X = team_rating_data_new[['win_loss_pct']]

y = team_rating_data_new['toward_500']

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X = sm.add_constant(X)
# model = LinearRegression(max_iter=10000)  # Increased max_iter helps the solver converge
# model.fit(X_train, y_train)

# y_pred = model.predict(X_test)         # Predicts hard labels (0 or 1)
# y_probs = model.predict_proba(X_test)  # Predicts continuous probabilities

# Fit and summarize OLS model
mod = sm.OLS( y,X)

res = mod.fit()

print(res.summary())


                            OLS Regression Results                            
Dep. Variable:             toward_500   R-squared:                       0.197
Model:                            OLS   Adj. R-squared:                  0.196
Method:                 Least Squares   F-statistic:                     188.7
Date:                Sun, 04 Oct 2026   Prob (F-statistic):           1.53e-38
Time:                        09:54:28   Log-Likelihood:                 541.66
No. Observations:                 769   AIC:                            -1079.
Df Residuals:                     767   BIC:                            -1070.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.1986      0.015     13.149   

In [30]:
import plotly.graph_objects as go
fig = px.scatter(team_rating_data_new,template='plotly_dark',x='win_loss_pct',y='toward_500',height=800)

x_vals = np.linspace(.0000, 1.0000, num=100)

y_vals =  0.1986  +  -0.3970  * x_vals

# 4. Append the equation trace to your Express figure
fig.add_trace(
    go.Scatter(
        x=x_vals,
        y=y_vals,
        mode="lines",
        name="0.2025 + -0.4009x",
        line=dict(color='#636EFA',width=5)
    )
)

fig.show()